# Mitochondrial network colocalisation — rho+ vs rho0

Image analysis accompanying the manuscript *[TODO: title]*.

Quantifies colocalisation between two fluorescent markers along the
mitochondrial network in *S. cerevisiae*, comparing wild-type (`rho+`) and
mtDNA-deficient (`rho0`) cells.

## Approach

The mitochondrial network is skeletonised in 3D with **MitoGraph**, which reduces the network to a set of
coordinates tracing its centreline. Intensities from both channels are then
sampled in a 3×3×3 voxel neighbourhood around each skeleton node, so
colocalisation is measured **within the mitochondrial network** rather than
across the whole cell volume — this avoids cytosolic and background signal
dominating the coefficients.

Two measures are computed per cell:

- **Manders' M1** — the fraction of Pda1-NG signal at positions where Su9-mKate2 is above threshold. Asymmetric: M1 and M2 differ, and only M1 is reported here.
- **Pearson correlation** — linear correlation of the two channels' intensities across all skeleton nodes, computed without thresholding.

Thresholds are set per cell by Li minimum cross-entropy, so each cell is
thresholded independently.


## Statistics

Comparisons are made on **replicate means (n = 3 per genotype)**, not on pooled
individual cells. Cells imaged in one session are not independent observations,
so testing on pooled cells would treat technical sampling as biological
replication and inflate significance. Figures are SuperPlots: individual cells
shown semi-transparent, replicate means as large points, and the test performed
on the replicate means.

Test: unpaired two-tailed Student's *t*-test. Significance shown as
`n.s.` / `*` / `**` / `***`.

## Running this notebook

**MitoGraph is run separately** on a Linux workstation; this notebook consumes
its output. The two stages were run on different machines, so the notebook does
not execute end to end from a single environment.

Set `IMAGE_DIR` (image folders), `DATA_DIR` (cached tables), and `OUT_DIR`
(figures) to the appropriate locations.

**Requires:** `pandas`, `numpy`, `scipy`, `scikit-image`, `imageio`, `seaborn`,
`matplotlib`, plus the MitoGraph binary on `PATH`.


## 1. Setup

Library imports and display settings for the image analysis pipeline.

| Library | Role in this notebook |
|---|---|
| `imageio` | Reading microscopy image files |
| `numpy` | Array operations on image data |
| `pandas` | Per-cell and per-object measurement tables |
| `skimage.filters.threshold_li` | Li minimum cross-entropy thresholding for segmentation |
| `scipy.ndimage` | Morphological operations on binary masks |
| `seaborn`, `matplotlib` | Figure generation |
| `os`, `shutil`, `subprocess` | File handling and external tool invocation |
| `pickle` | Serialising intermediate results |

Pandas display options are widened so that per-cell measurement tables print in
full during interactive inspection.

In [ ]:
import os
import numpy as np
import pandas as pd
import imageio
import seaborn as sns
import matplotlib.pyplot as plt
import shutil
import subprocess
import pickle
from skimage.filters import threshold_li
import scipy
from scipy.ndimage.morphology import binary_dilation
pd.set_option('display.max_rows', 700)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)


## 2. Image analysis functions

Extracts per-voxel intensities along the mitochondrial network skeleton
computed by MitoGraph, for colocalisation analysis between channels.

**`runMitoGraph`** — invokes the MitoGraph binary on a folder of TIF stacks with
voxel dimensions xy = 0.07 µm, z = 0.200 µm, then moves its output files into a
`MitoGraphFiles` subfolder.

**`convertIntoPixelUnit`** — reads MitoGraph's skeleton coordinate table and
converts µm coordinates to voxel indices by dividing by the voxel size and
rounding.

**`readIntensitiesInMitoNetwork` / `getIntensities`** — for each skeleton
coordinate, samples channels 2 and 3 and records the mean intensity of the
surrounding 3×3×3 voxel neighbourhood. Images are flipped along the y axis to
match MitoGraph's coordinate convention.

**`main`** — iterates over the Channel 1 TIF files in a folder, building a
per-cell table of skeleton coordinates with matched intensities in the other
channels.

**`getMandersColCoeff`** — Manders' M1: the fraction of total signal in one
channel that falls within the thresholded mask of the other.

**`thresholdScan`** — Li minimum cross-entropy threshold, optionally followed by
one round of binary dilation, returning the intensity-weighted masked signal.

**`get_pvalue` / `get_pvalue_fl`** — Welch-style independent *t*-test comparing
each mutant against the `rho+` reference, returning the p-value as string or
float.

In [ ]:
def main(folder):
    all_cells = {}
    for file in os.listdir(folder + 'Channel_1/'):
        
        if file.endswith(".tif"):
            filename = file[:-9]
            print(file , "+", file[:-9])
            df = convertIntoPixelUnit(folder, filename, 2, file[-7:-4])
            df = readIntensitiesInMitoNetwork(df, folder + 'Channel_', file)
            all_cells[file] = df
    return all_cells

def runMitoGraph(folder):
    '''takes a folder name as input and runs the MitoGraph software on this folder '''
    os.chdir('/home/huygens/Documents/Microscopy')    
    command = 'MitoGraph -xy 0.07 -z 0.200 -path ' + folder
    
    output=subprocess.check_output(command, stderr=subprocess.STDOUT, shell=True)
    os.mkdir(os.path.join(folder,'MitoGraphFiles'))
    
    for file in os.listdir(folder):
        if not file.endswith('.tif') and file != 'MitoGraphFiles':
            shutil.move(folder + '/' + file, folder + '/MitoGraphFiles/')
            
    return output  


def readIntensitiesInMitoNetwork(df, folder, filename2, xydim=0.11, zdim=0.2):
    for j in [2,3]:
        file2 = folder + str(j) + '/' + filename2[:-9] + str(j) + filename2[-8:-4] + '.tif'
        df['Intensity ' + str(j)] = getIntensities(file2, j, df)
    return df
    
def getIntensities(file, channel, df):
    im = np.flip(np.array(imageio.mimread(file)),1)
    return [im[df['alt_z'][i]-1:df['alt_z'][i]+2, df['alt_y'][i]-1:df['alt_y'][i]+2,df['alt_x'][i]-1:df['alt_x'][i]+2].mean() for i in range(df.shape[0])]
    
def convertIntoPixelUnit(folder, filename, channel, cellNumber, xydim=0.11, zdim=0.2):
    df = pd.read_table(folder + "Channel_" + str(channel) + '/MitoGraphFiles/' + filename + str(channel) + '_'+ cellNumber + '.txt')
    # the following converts the µm unit into pixel unit
    df['alt_x'] = round(df['x']/xydim).astype('int32')
    df['alt_y'] = round(df['y']/xydim).astype('int32')
    df['alt_z'] = round(df['z']/zdim).astype('int32')
    return df



def getMandersColCoeff(array_one, array_two):
    numerator = sum([array_one[i] for i in range(array_one.size) if array_two[i] > 0])
    denominator = array_one.sum()
    m1 = numerator/denominator
    return m1



def thresholdScan(lineScan, dilate=True):    
    thresh = threshold_li(lineScan)
    binary = lineScan > thresh
    
    if dilate:
        binary = binary_dilation(binary)
    
    return lineScan * binary

def get_pvalue(df, mutant):
    #statistic, pvalue = scipy.stats.mannwhitneyu(df['wt'], df[mutant])
    statistic, pvalue = scipy.stats.ttest_ind(df["rho+"], df[mutant], nan_policy='omit')
    return str(float(round(pvalue, 6)))
def get_pvalue_fl(df, mutant):
    #statistic, pvalue = scipy.stats.mannwhitneyu(df['wt'], df[mutant])
    statistic, pvalue = scipy.stats.ttest_ind(df["rho+"], df[mutant], nan_policy='omit')
    return float(round(pvalue, 6))

## 3. Run MitoGraph

Runs MitoGraph on the Channel 2 image stacks to skeletonise the mitochondrial
network in 3D. Output files are written alongside the images and then moved into
a `MitoGraphFiles` subfolder.

This step calls an external binary and is run once per dataset; downstream cells
read its output tables.

**Requires:** MitoGraph on the system `PATH`.
Voxel dimensions passed: xy = 0.07 µm, z = 0.200 µm.

In [ ]:
# Directory containing the per-channel image folders (Channel_1/, Channel_2/, ...).
IMAGE_DIR = '.'
runMitoGraph(f'{IMAGE_DIR}/Channel_2')

## 4. Extract per-cell skeleton measurements

Runs the extraction over every cell in the dataset. `main` iterates the Channel 1
TIF files, reads the matching MitoGraph skeleton table for Channel 2, converts
its µm coordinates to voxel indices, and samples the mean intensity of Channels
2 and 3 in a 3×3×3 neighbourhood around each skeleton node.

The result is a dictionary keyed by filename, each value a per-node table for one
cell: skeleton coordinates plus matched intensities in both channels.



In [ ]:
# Root folder containing Channel_1/, Channel_2/, Channel_3/ subfolders.
folder = f"{IMAGE_DIR}/"
all_cells = main(folder)

In [ ]:
all_cells = main(folder)

180324_978_rep2_1_1_005.tif + 180324_978_rep2_1_
180324_978_rep2_1_1_006.tif + 180324_978_rep2_1_
180324_978_rep2_1_1_007.tif + 180324_978_rep2_1_
180324_978_rep2_1_1_008.tif + 180324_978_rep2_1_
180324_978_rep2_1_1_009.tif + 180324_978_rep2_1_
180324_978_rep2_1_1_010.tif + 180324_978_rep2_1_
180324_978_rep2_1_1_011.tif + 180324_978_rep2_1_
180324_978_rep2_1_1_012.tif + 180324_978_rep2_1_
180324_978_rep2_1_1_013.tif + 180324_978_rep2_1_
180324_978_rep2_1_1_014.tif + 180324_978_rep2_1_
180324_978_rep2_1_1_015.tif + 180324_978_rep2_1_
180324_978_rep2_1_1_017.tif + 180324_978_rep2_1_
180324_978_rep2_1_1_019.tif + 180324_978_rep2_1_
180324_978_rep2_1_1_028.tif + 180324_978_rep2_1_
180324_978_rep2_1_1_029.tif + 180324_978_rep2_1_
180324_978_rep2_1_1_073.tif + 180324_978_rep2_1_
180324_978_rep2_1_1_074.tif + 180324_978_rep2_1_
180324_978_rep2_1_1_075.tif + 180324_978_rep2_1_
180324_978_rep2_1_1_076.tif + 180324_978_rep2_1_
180324_978_rep2_1_1_077.tif + 180324_978_rep2_1_
180324_978_rep2_1_1_

180324_978_rep2_4_1_078.tif + 180324_978_rep2_4_
180324_978_rep2_4_1_140.tif + 180324_978_rep2_4_
180324_978_rep2_4_1_141.tif + 180324_978_rep2_4_
180324_978_rep2_4_1_142.tif + 180324_978_rep2_4_
180324_978_rep2_4_1_143.tif + 180324_978_rep2_4_
180324_978_rep2_4_1_144.tif + 180324_978_rep2_4_
180324_978_rep2_4_1_145.tif + 180324_978_rep2_4_
180324_978_rep2_4_1_147.tif + 180324_978_rep2_4_
180324_978_rep2_4_1_148.tif + 180324_978_rep2_4_
180324_978_rep2_4_1_149.tif + 180324_978_rep2_4_
180324_978_rep2_4_1_150.tif + 180324_978_rep2_4_
180324_978_rep2_4_1_152.tif + 180324_978_rep2_4_
20240325_yCO987_rep3_0_1_067.tif + 20240325_yCO987_rep3_0_
20240325_yCO987_rep3_0_1_073.tif + 20240325_yCO987_rep3_0_
20240325_yCO987_rep3_0_1_074.tif + 20240325_yCO987_rep3_0_
20240325_yCO987_rep3_0_1_076.tif + 20240325_yCO987_rep3_0_
20240325_yCO987_rep3_0_1_079.tif + 20240325_yCO987_rep3_0_
20240325_yCO987_rep3_0_1_080.tif + 20240325_yCO987_rep3_0_
20240325_yCO987_rep3_0_1_081.tif + 20240325_yCO987_rep3_0_

20240325_yCO987_rep3_3_1_012.tif + 20240325_yCO987_rep3_3_
20240325_yCO987_rep3_3_1_013.tif + 20240325_yCO987_rep3_3_
20240325_yCO987_rep3_3_1_014.tif + 20240325_yCO987_rep3_3_
20240325_yCO987_rep3_3_1_015.tif + 20240325_yCO987_rep3_3_
20240325_yCO987_rep3_3_1_016.tif + 20240325_yCO987_rep3_3_
20240325_yCO987_rep3_3_1_017.tif + 20240325_yCO987_rep3_3_
20240325_yCO987_rep3_3_1_018.tif + 20240325_yCO987_rep3_3_
20240325_yCO987_rep3_3_1_051.tif + 20240325_yCO987_rep3_3_
20240325_yCO987_rep3_3_1_052.tif + 20240325_yCO987_rep3_3_
20240325_yCO987_rep3_3_1_054.tif + 20240325_yCO987_rep3_3_
20240325_yCO987_rep3_3_1_056.tif + 20240325_yCO987_rep3_3_
20240325_yCO987_rep3_3_1_058.tif + 20240325_yCO987_rep3_3_
20240325_yCO987_rep3_3_1_059.tif + 20240325_yCO987_rep3_3_
20240325_yCO987_rep3_3_1_061.tif + 20240325_yCO987_rep3_3_
20240325_yCO987_rep3_3_1_062.tif + 20240325_yCO987_rep3_3_
20240325_yCO987_rep3_3_1_064.tif + 20240325_yCO987_rep3_3_
20240325_yCO987_rep3_3_1_069.tif + 20240325_yCO987_rep3_

## 5. Per-cell colocalisation coefficients

Computes two colocalisation measures per cell from the intensities sampled along
the mitochondrial skeleton, producing one row per cell.

- **Manders' M1** — the fraction of Channel 2 signal that falls at skeleton
  positions where Channel 3 is above its threshold. Both channels are
  thresholded with Li minimum cross-entropy first (`dilate=False` here, so no
  mask expansion is applied).
- **Pearson correlation** — linear correlation between the two channels'
  intensities across all skeleton nodes, computed on the unthresholded values.

The two are complementary: Pearson measures whether the intensities co-vary,
Manders measures what fraction of one signal overlaps the other, and they can
disagree when one channel is present at a roughly constant level.

In [ ]:
results = pd.DataFrame()
for key in all_cells.keys():
    temp_df = all_cells[key]
    correlation_df = {}
    correlation_df['Cell'] = key
    array_one = np.array(temp_df['Intensity 2'])
    array_two = np.array(temp_df['Intensity 3'])
    one = thresholdScan(array_one, dilate = False)
    two = thresholdScan(array_two, dilate = False)
    correlation_df['_manders'] = (getMandersColCoeff(one, two))
    correlation_df['_pearson'] = temp_df['Intensity 2'].corr(temp_df['Intensity 3'])
    results = results.append(correlation_df, ignore_index=True)
results.index = results['Cell']
results = results.iloc[:, 1:]
df2 = results

C:\Users\Thoma\AppData\Local\Temp\ipykernel_101420\374235864.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append(correlation_df, ignore_index=True)
C:\Users\Thoma\AppData\Local\Temp\ipykernel_101420\374235864.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append(correlation_df, ignore_index=True)
C:\Users\Thoma\AppData\Local\Temp\ipykernel_101420\374235864.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append(correlation_df, ignore_index=True)
C:\Users\Thoma\AppData\Local\Temp\ipykernel_101420\374235864.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results 

C:\Users\Thoma\AppData\Local\Temp\ipykernel_101420\374235864.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append(correlation_df, ignore_index=True)
C:\Users\Thoma\AppData\Local\Temp\ipykernel_101420\374235864.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append(correlation_df, ignore_index=True)
C:\Users\Thoma\AppData\Local\Temp\ipykernel_101420\374235864.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append(correlation_df, ignore_index=True)
C:\Users\Thoma\AppData\Local\Temp\ipykernel_101420\374235864.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results 

C:\Users\Thoma\AppData\Local\Temp\ipykernel_101420\374235864.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append(correlation_df, ignore_index=True)
C:\Users\Thoma\AppData\Local\Temp\ipykernel_101420\374235864.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append(correlation_df, ignore_index=True)
C:\Users\Thoma\AppData\Local\Temp\ipykernel_101420\374235864.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append(correlation_df, ignore_index=True)
C:\Users\Thoma\AppData\Local\Temp\ipykernel_101420\374235864.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results 

C:\Users\Thoma\AppData\Local\Temp\ipykernel_101420\374235864.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append(correlation_df, ignore_index=True)
C:\Users\Thoma\AppData\Local\Temp\ipykernel_101420\374235864.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append(correlation_df, ignore_index=True)
C:\Users\Thoma\AppData\Local\Temp\ipykernel_101420\374235864.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append(correlation_df, ignore_index=True)
C:\Users\Thoma\AppData\Local\Temp\ipykernel_101420\374235864.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results 

In [ ]:
def repetition(x):
    if 'rep3' in x:
        return 'rep 3'
    elif 'rep2' in x:
        return 'rep 2'
    elif 'rep1' in x:
        return 'rep 1'

In [ ]:
df2["filname"]=df2.index.to_list()

In [ ]:
df2['Replicate']= df2['filname'].apply(repetition)

## 6. Load per-genotype colocalisation datasets

The extraction pipeline above was run separately for each genotype and the
per-cell results cached, so this analysis can be re-run without repeating
MitoGraph and the intensity sampling. This cell loads the two cached tables,
which are then annotated with genotype and biological replicate and merged.

- `df2` — [TODO: wild-type / rho+ cells]
- `rho` — rho0 cells (mtDNA-deficient)

Each table has one row per cell, indexed by source filename, with Manders' M1
and Pearson coefficients.

In [ ]:
rho = pd.read_pickle(f"{DATA_DIR}/coloc_pda_rho0.pkl")
df2 = pd.read_pickle(f"{DATA_DIR}/coloc_pda_wt.pkl")

In [ ]:
df = pd.concat([df2,rho])

## 7. Manders coefficient — wild type vs rho0

SuperPlot comparing colocalisation between genotypes, showing individual cells,
replicate means, and the genotype summary in one panel.

**Layers:**

| Layer | Data | Appearance |
|---|---|---|
| Individual cells | `b` — every cell | Small, semi-transparent, coloured by replicate |
| Replicate means | `c` — mean per genotype × replicate | Large, black-outlined, same replicate colours |
| Genotype summary | `d` — mean of replicate means | Box |
| Error bars | `e` — SD across replicate means | Black caps |

**Statistics.** The test is performed on **replicate means, n = 3 per genotype**,
not on individual cells. This is the appropriate unit of replication: cells
within one imaging session are not independent observations, and testing on
pooled cells would inflate significance by treating technical sampling as
biological replication. Significance is annotated as `n.s.` / `*` / `**` / `***`.

In [ ]:
b = df

c = b.groupby(['Genotype','Replicate'], as_index=False).mean()

d = c.groupby('Genotype', as_index=False).mean()
d.index = d['Genotype']
d = d.T[['rho+', 'rho0']].T
e = c.groupby('Genotype', as_index=False).std()
e.index = e['Genotype']
e = e.T[['rho+','rho0' ]].T
f= c.pivot_table(columns='Genotype', values='_manders', index="Replicate") 

In [ ]:
d

In [ ]:
rho.shape[0]

In [ ]:
df2.shape[0]

In [ ]:
nupur2 = ["#BD5B28","#FFD30A","#4d6171","#162734","#b6c9c1","#88A27D"]

In [ ]:
df.to_pickle(f"{DATA_DIR}/coloc_pda_rho+_and_rho0.pkl")


In [ ]:
plt.figure()
plt.rcParams['svg.fonttype']='none'
fig, ax = plt.subplots(figsize=(7,6))
u = sns.color_palette('twilight_r',5)
sns.swarmplot(x='Genotype', y='_manders', hue='Replicate',
              order=['rho+','rho0'],
              hue_order=['rep 1','rep 2', 'rep 3'],data = b, alpha=0.3, size=4, palette=nupur2, ax=ax)

sns.swarmplot(x='Genotype', y='_manders', hue='Replicate',
              order=['rho+', 'rho0'],
              hue_order=['rep 1','rep 2', 'rep 3'],data = c, alpha=0.8, size=10,
              edgecolor='k', linewidth=1,palette=nupur2, ax=ax)
#sns.swarmplot(x='Genotype', y='ATP6 ratio to origin cell', hue='experiment', data = c, size=15, edgecolor='k', linewidth=1, ax=ax)
sns.boxplot(x='Genotype', y='_manders',  width=0.4,data = d, ax=ax)
plt.errorbar(range(2), d['_manders'], yerr=e['_manders'], fmt='none',capsize=5,ecolor='k' )
ax.get_legend().remove()
ax.set_ylim(0,1.2)
ax.set(xlabel="Strain", ylabel="Manders coefficient")



for i,j in enumerate(["rho0"]):

    x1, x2, y, h = 0, 1+i, 1.1+i*.3, .01
    plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.2, c='k')
    #print(get_pvalue_fl(f,j))
    if get_pvalue_fl(f,j) > .05:
        print("ns :" + str(get_pvalue_fl(f,j)))        
        plt.text((x1+x2)*.5, y+h*1.5, "n.s.", ha='center', va='bottom', color='k',size = 22)
    elif get_pvalue_fl(f,j)<.05 and get_pvalue_fl(f,j)>.01:
        print("uno"+str(get_pvalue_fl(f,j)))
        plt.text((x1+x2)*.5, y+h*1.5,  "*", ha='center', va='bottom', color='k',size = 30)
    elif get_pvalue_fl(f,j)<.01 and get_pvalue_fl(f,j)>.001 :
        print("zwo"+str(get_pvalue_fl(f,j)))
        plt.text((x1+x2)*.5, y+h-0.001, "**", ha='center', va='bottom', color='k',size = 15)
    elif get_pvalue_fl(f,j)<.001:
        print("drei " + str(get_pvalue_fl(f,j)))
        plt.text((x1+x2)*.5, y+h*.5,  "***", ha='center', va='bottom', color='k',size = 30)
    else:
        plt.text((x1+x2)*.5, y+h*1.5, "P =  "+get_pvalue(f,j), ha='center', va='bottom', color='k')

# plt.savefig(f"{OUT_DIR}/WTvsRho0_Manders.svg", bbox_inches="tight", dpi=300)